<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-LLM-Document-Intelligence-Platform/blob/main/Enterprise_LLM_Document_Intelligence_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
from datetime import datetime
import random
import json
import re

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

BASE_PATH = Path('/content/enterprise_llm_document_intelligence_platform')
RAW_PATH = BASE_PATH / 'data' / 'raw_documents'
PROCESSED_PATH = BASE_PATH / 'data' / 'processed_documents'
SEARCH_READY_PATH = BASE_PATH / 'data' / 'search_ready'
OUTPUT_PATH = BASE_PATH / 'outputs'
CONFIG_PATH = BASE_PATH / 'config'

for p in [RAW_PATH, PROCESSED_PATH, SEARCH_READY_PATH, OUTPUT_PATH, CONFIG_PATH]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)

print('Project folders created successfully.')
print(BASE_PATH)

Project folders created successfully.
/content/enterprise_llm_document_intelligence_platform


In [ ]:
extraction_schema = {
    'document_types': [
        'quality_report',
        'supplier_invoice',
        'maintenance_record',
        'engineering_change_notice'
    ],
    'fields_to_extract': [
        'document_id',
        'plant_code',
        'supplier_name',
        'invoice_amount',
        'issue_type',
        'equipment_id',
        'change_order_id',
        'document_date'
    ]
}

with open(CONFIG_PATH / 'extraction_schema.json', 'w') as f:
    json.dump(extraction_schema, f, indent=2)

print(json.dumps(extraction_schema, indent=2))

{
  "document_types": [
    "quality_report",
    "supplier_invoice",
    "maintenance_record",
    "engineering_change_notice"
  ],
  "fields_to_extract": [
    "document_id",
    "plant_code",
    "supplier_name",
    "invoice_amount",
    "issue_type",
    "equipment_id",
    "change_order_id",
    "document_date"
  ]
}


In [ ]:
documents = []

supplier_names = ['Magna', 'Bosch', 'Denso', 'Lear', 'Aptiv']
plants = ['PLT100', 'PLT200', 'PLT300', 'PLT400']
issue_types = ['weld defect', 'paint inconsistency', 'sensor failure', 'material shortage', 'assembly variance']
equipment_ids = ['EQP1001', 'EQP1002', 'EQP1003', 'EQP1004']

for i in range(1, 501):
    doc_type = random.choice(extraction_schema['document_types'])
    plant = random.choice(plants)
    supplier = random.choice(supplier_names)
    issue = random.choice(issue_types)
    equipment = random.choice(equipment_ids)
    amount = round(float(np.random.normal(12500, 3500)), 2)
    doc_date = f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    change_order_id = f"ECO-{random.randint(10000,99999)}"

    if doc_type == 'quality_report':
        text = f"Document ID DOC{i:05d}. Plant {plant}. Quality inspection found {issue} on equipment {equipment}. Document date {doc_date}. Supplier reference {supplier}."
    elif doc_type == 'supplier_invoice':
        text = f"Document ID DOC{i:05d}. Supplier invoice from {supplier} for plant {plant}. Invoice amount ${amount}. Document date {doc_date}. Material issue related to {issue}."
    elif doc_type == 'maintenance_record':
        text = f"Document ID DOC{i:05d}. Maintenance record for equipment {equipment} at plant {plant}. Reported issue {issue}. Service date {doc_date}. Vendor {supplier}."
    else:
        text = f"Document ID DOC{i:05d}. Engineering change notice {change_order_id} for plant {plant}. Change related to {issue}. Effective date {doc_date}. Supplier {supplier}."

    documents.append({
        'document_id': f'DOC{i:05d}',
        'document_type': doc_type,
        'document_text': text
    })

documents_df = pd.DataFrame(documents)
documents_df.to_csv(RAW_PATH / 'enterprise_documents.csv', index=False)
print(documents_df.head())

  document_id   document_type  \
0    DOC00001  quality_report   
1    DOC00002  quality_report   
2    DOC00003  quality_report   
3    DOC00004  quality_report   
4    DOC00005  quality_report   

                                       document_text  
0  Document ID DOC00001. Plant PLT100. Quality in...  
1  Document ID DOC00002. Plant PLT400. Quality in...  
2  Document ID DOC00003. Plant PLT200. Quality in...  
3  Document ID DOC00004. Plant PLT200. Quality in...  
4  Document ID DOC00005. Plant PLT100. Quality in...  


In [10]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

documents_df = pd.read_csv(RAW_PATH / 'enterprise_documents.csv')
documents_df['normalized_text'] = documents_df['document_text'].apply(normalize_text)
documents_df.to_csv(PROCESSED_PATH / 'processed_documents.csv', index=False)
print('Processed documents saved successfully.')

Processed documents saved successfully.


In [11]:
X = documents_df['normalized_text']
y = documents_df['document_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)
preds = clf.predict(X_test_vec)

accuracy = accuracy_score(y_test, preds)
print('Classification Accuracy:', round(accuracy, 4))
print(classification_report(y_test, preds))

classification_results = pd.DataFrame({
    'document_text': X_test.values,
    'actual_type': y_test.values,
    'predicted_type': preds
})
classification_results.to_csv(OUTPUT_PATH / 'classification_results.csv', index=False)

Classification Accuracy: 1.0
                           precision    recall  f1-score   support

engineering_change_notice       1.00      1.00      1.00        29
       maintenance_record       1.00      1.00      1.00        26
           quality_report       1.00      1.00      1.00        21
         supplier_invoice       1.00      1.00      1.00        24

                 accuracy                           1.00       100
                macro avg       1.00      1.00      1.00       100
             weighted avg       1.00      1.00      1.00       100



In [ ]:
def extract_field(pattern, text):
    match = re.search(pattern, text, re.IGNORECASE)
    return match.group(1) if match else None

extracted_rows = []
for _, row in documents_df.iterrows():
    text = row['document_text']
    extracted_rows.append({
        'document_id': row['document_id'],
        'document_type': row['document_type'],
        'plant_code': extract_field(r'Plant\s+(PLT\d+)|plant\s+(PLT\d+)', text),
        'supplier_name': extract_field(r'Supplier(?:\s+invoice\s+from|\s+reference|\s+|\s+.*?from\s+)(Magna|Bosch|Denso|Lear|Aptiv)', text),
        'invoice_amount': extract_field(r'\$(\d+\.?\d*)', text),
        'equipment_id': extract_field(r'equipment\s+(EQP\d+)', text),
        'change_order_id': extract_field(r'(ECO-\d+)', text),
        'document_date': extract_field(r'(\d{4}-\d{2}-\d{2})', text)
    })

extracted_df = pd.DataFrame(extracted_rows)

# fix regex tuples for plant extraction
if 'plant_code' in extracted_df.columns:
    extracted_df['plant_code'] = extracted_df['plant_code'].apply(lambda x: x[0] if isinstance(x, tuple) and x[0] else (x[1] if isinstance(x, tuple) and len(x) > 1 else x))

extracted_df.to_csv(OUTPUT_PATH / 'extracted_entities.csv', index=False)
print(extracted_df.head())

  document_id   document_type plant_code supplier_name invoice_amount  \
0    DOC00001  quality_report     PLT100          None           None   
1    DOC00002  quality_report     PLT400          None           None   
2    DOC00003  quality_report     PLT200          None           None   
3    DOC00004  quality_report     PLT200          None           None   
4    DOC00005  quality_report     PLT100          None           None   

  equipment_id change_order_id document_date  
0      EQP1002            None    2024-03-24  
1      EQP1001            None    2024-04-08  
2      EQP1002            None    2024-08-19  
3      EQP1003            None    2024-03-07  
4      EQP1003            None    2024-06-20  


In [ ]:
search_ready_df = documents_df[['document_id', 'document_type', 'document_text']].copy()
search_ready_df['semantic_record'] = (
    'DocumentType: ' + search_ready_df['document_type'] + ' | ' + search_ready_df['document_text']
)
search_ready_df.to_csv(SEARCH_READY_PATH / 'semantic_search_records.csv', index=False)
search_ready_df.to_csv(OUTPUT_PATH / 'semantic_search_records.csv', index=False)
print(search_ready_df.head())

  document_id   document_type  \
0    DOC00001  quality_report   
1    DOC00002  quality_report   
2    DOC00003  quality_report   
3    DOC00004  quality_report   
4    DOC00005  quality_report   

                                       document_text  \
0  Document ID DOC00001. Plant PLT100. Quality in...   
1  Document ID DOC00002. Plant PLT400. Quality in...   
2  Document ID DOC00003. Plant PLT200. Quality in...   
3  Document ID DOC00004. Plant PLT200. Quality in...   
4  Document ID DOC00005. Plant PLT100. Quality in...   

                                     semantic_record  
0  DocumentType: quality_report | Document ID DOC...  
1  DocumentType: quality_report | Document ID DOC...  
2  DocumentType: quality_report | Document ID DOC...  
3  DocumentType: quality_report | Document ID DOC...  
4  DocumentType: quality_report | Document ID DOC...  


In [ ]:
semantic_vectorizer = TfidfVectorizer(stop_words='english')
semantic_matrix = semantic_vectorizer.fit_transform(search_ready_df['semantic_record'])

query = 'quality issue on equipment in plant PLT200'
query_vec = semantic_vectorizer.transform([query])
similarity_scores = cosine_similarity(query_vec, semantic_matrix).flatten()

top_indices = similarity_scores.argsort()[-5:][::-1]
search_results = search_ready_df.iloc[top_indices].copy()
search_results['similarity_score'] = similarity_scores[top_indices]
search_results

,document_id,document_type,document_text,semantic_record,similarity_score
297,DOC00298,quality_report,Document ID DOC00298. Plant PLT200. Quality in...,DocumentType: quality_report | Document ID DOC...,0.321524
469,DOC00470,quality_report,Document ID DOC00470. Plant PLT200. Quality in...,DocumentType: quality_report | Document ID DOC...,0.320251
282,DOC00283,quality_report,Document ID DOC00283. Plant PLT200. Quality in...,DocumentType: quality_report | Document ID DOC...,0.319932
329,DOC00330,quality_report,Document ID DOC00330. Plant PLT200. Quality in...,DocumentType: quality_report | Document ID DOC...,0.319629
246,DOC00247,quality_report,Document ID DOC00247. Plant PLT200. Quality in...,DocumentType: quality_report | Document ID DOC...,0.319402
